# Toy RAG --- TF-IDF retrieve + extractive generate

Interactive companion to `rag.py` / `run_smoke.py`.

Walk through: build the TF-IDF index, retrieve, extractive answer, compare baseline.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from corpus import DOCUMENTS, QA_PAIRS
from rag import (
    build_vocab, compute_idf, tfidf_matrix, l2_normalize,
    retrieve, extractive_generate, baseline_generate, evaluate_qa,
)

print(f'docs={len(DOCUMENTS)} qa={len(QA_PAIRS)}')
print(DOCUMENTS[0])

In [ ]:
word2id, id2word = build_vocab(DOCUMENTS)
idf = compute_idf(DOCUMENTS, word2id)
X = l2_normalize(tfidf_matrix(DOCUMENTS, word2id, idf))
print(f'vocab={len(id2word)} matrix={X.shape}')

In [ ]:
q = QA_PAIRS[0]['question']
gold = QA_PAIRS[0]['gold_answer']
hits = retrieve(q, word2id, idf, X, k=3)
print('Q:', q)
print('gold:', gold)
for h in hits:
    print(f"  #{h['rank']} {h['doc_id']} cos={h['cosine']:.3f} :: {h['text'][:80]}...")
print()
print('RAG:', extractive_generate(q, hits))
print('BASE:', baseline_generate(q, mode='fixed'))

In [ ]:
out = evaluate_qa(word2id, idf, X, QA_PAIRS, DOCUMENTS, k=3, seed=42)
for key in ('hit_at_1', 'hit_at_3', 'mrr', 'rag_answer_accuracy',
            'baseline_answer_accuracy', 'faithfulness'):
    print(f'{key}: {out[key]:.4f}')

## Takeaway

When Hit@k is strong, extractive generation usually contains the gold span;
a no-retrieval baseline almost never does. That gap is the point of RAG.